In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import Window

# Caminhos do Bronze
BRONZE = "/Volumes/workspace/olist/bronze"
SILVER = "/Volumes/workspace/olist/silver"

# Lê as 4 tabelas do Bronze
df_orders      = spark.read.format("delta").load(f"{BRONZE}/orders")
df_order_items = spark.read.format("delta").load(f"{BRONZE}/orders_items")
df_customers   = spark.read.format("delta").load(f"{BRONZE}/customers")
df_products    = spark.read.format("delta").load(f"{BRONZE}/products")

# Confirma leitura
print(f"orders:      {df_orders.count()} registros")
print(f"order_items: {df_order_items.count()} registros")
print(f"customers:   {df_customers.count()} registros")
print(f"products:    {df_products.count()} registros")

In [0]:
# ORDERS - remove colunas de auditoria, padroniza status
df_orders_silver = df_orders \
    .drop("_ingest_timestamp", "_source_filename", "_processing_date", "_execucao_id") \
    .filter(F.col("order_id").isNotNull()) \
    .filter(F.col("customer_id").isNotNull()) \
    .withColumn("order_status", F.lower(F.trim(F.col("order_status"))))

# ORDER_ITEMS - cast de valores monetários
df_order_items_silver = df_order_items \
    .drop("_ingest_timestamp", "_source_filename", "_processing_date", "_execucao_id") \
    .filter(F.col("order_id").isNotNull()) \
    .withColumn("price", F.col("price").cast("double")) \
    .withColumn("freight_value", F.col("freight_value").cast("double"))

#CUSTOMERS -padroniza cidade e estado
df_customers_silver = df_customers \
    .drop("_ingest_timestamp", "_source_filename", "_processing_date", "_execucao_id") \
    .filter(F.col("customer_id").isNotNull()) \
    .withColumn("customer_city", F.lower(F.trim(F.col("customer_city")))) \
    .withColumn("customer_state", F.lower(F.trim(F.col("customer_state"))))

# PRODUCTS - remove colunas de auditoria
df_products_silver = df_products \
    .drop("ingest_timestamp", "source_filename", "processing_date", "execucao_id") \
    .filter(F.col("product_id").isNotNull()) \
    .withColumn("product_category_name", F.coalesce(F.col("product_category_name"), F.lit("unknown")))

print("Limpeza concluída!")
print(f"orders_silver:      {df_orders_silver.count()} registros")
print(f"order_items_silver: {df_order_items_silver.count()} registros")
print(f"customers_silver:   {df_customers_silver.count()} registros")
print(f"products_silver:    {df_products_silver.count()} registros")

In [0]:
# Join principal - uma linha por item de pedido
df_silver = df_order_items_silver \
    .join(df_orders_silver, on="order_id", how="left") \
    .join(df_customers_silver, on="customer_id", how="left") \
    .join(df_products_silver, on="product_id", how="left")

print(f"Total de registros: {df_silver.count()}")
print(f"Total de colunas:   {len(df_silver.columns)}")
print(f"\nColunas: {df_silver.columns}")

In [0]:
# Remove colunas de auditoria e corrige typos
df_silver = df_silver \
    .drop("_ingest_timestamp", "_source_filename", "_processing_date", "_execucao_id") \
    .withColumnRenamed("product_name_lenght",        "product_name_length") \
    .withColumnRenamed("product_description_lenght", "product_description_length")

print(f"Total de registros: {df_silver.count()}")
print(f"Total de colunas:   {len(df_silver.columns)}")
print(f"\nColunas: {df_silver.columns}")

In [0]:
# Janelas de cálculo
window_pedido = Window.partitionBy("order_id")
window_cliente = Window.partitionBy("customer_unique_id") \
                    .orderBy("order_purchase_timestamp") \
                    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Etapa 1 - valor total do pedido
df_silver = df_silver \
    .withColumn("valor_total_pedido",
                F.sum(F.col("price") + F.col("freight_value")).over(window_pedido))
    
# Etapa 2 - rank do item do pedido
df_silver = df_silver \
    .withColumn("rank_item_pedido",
        F.dense_rank().over(Window.partitionBy("order_id").orderBy(F.col("price").desc())))
    
# Etapa 3 - LTV acumulado por cliente
df_silver = df_silver \
    .withColumn("ltv_acumulado",
        F.sum(F.col("price") + F.col("freight_value")).over(window_cliente))
    
print(f"Colunas após window functions: {len(df_silver.columns)}")
display(df_silver.select(
    "order_id",
    "customer_id",
    "price",
    "freight_value",
    "valor_total_pedido",
    "rank_item_pedido",
    "ltv_acumulado"
).limit(10))

In [0]:
from datetime import date as dt

df_silver \
    .withColumn("_processing_date", F.lit(str(dt.today()))) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_status", "_processing_date") \
    .save(f"{SILVER}/orders_enriched")

total = spark.read.format("delta").load(f"{SILVER}/orders_enriched").count()
print(f"Silver gravado: {total} registros")

In [0]:
import time
from pyspark.sql.types import (StructType, StructField, StringType, 
                                LongType, DoubleType, TimestampType)
from datetime import datetime as dt, date

LOG_PATH   = "/Volumes/workspace/olist/bronze/_log"
LOG_SCHEMA = StructType([
    StructField("execucao_id",        StringType(),    nullable=False),
    StructField("tabela",             StringType(),    nullable=False),
    StructField("status",             StringType(),    nullable=False),
    StructField("registros_lidos",    LongType(),      nullable=True),
    StructField("registros_gravados", LongType(),      nullable=True),
    StructField("divergencia_pct",    DoubleType(),    nullable=True),
    StructField("tempo_segundos",     DoubleType(),    nullable=True),
    StructField("mensagem",           StringType(),    nullable=True),
    StructField("timestamp",          TimestampType(), nullable=False)
])

def gravar_log(execucao_id, tabela, status, registros_lidos=None,
               registros_gravados=None, divergencia_pct=None,
               tempo_segundos=None, mensagem=None):

    linha = [{
        "execucao_id":        execucao_id,
        "tabela":             tabela,
        "status":             status,
        "registros_lidos":    registros_lidos,
        "registros_gravados": registros_gravados,
        "divergencia_pct":    divergencia_pct,
        "tempo_segundos":     tempo_segundos,
        "mensagem":           mensagem,
        "timestamp":          dt.now()
    }]

    spark.createDataFrame(linha, schema=LOG_SCHEMA) \
        .write \
        .format("delta") \
        .mode("append") \
        .save(LOG_PATH)

def processar_silver(execucao_id: str):
    inicio = time.time()
    nome   = "orders_enriched"

    print(f"\n{'='*50}")
    print(f"Processando Silver: {nome}")
    print(f"{'='*50}")

    try:
        # Lê o Silver gravado
        df = spark.read.format("delta").load(f"{VOLUMES['silver']}/{nome}")
        registros_lidos = df.count()
        print(f"  Lidos: {registros_lidos} registros")

        # Consolida todos os checks
        problemas = []

        # Check 1 — divergência contra Bronze
        registros_bronze = spark.read.format("delta") \
            .load(f"{VOLUMES['bronze']}/order_items").count()
        divergencia = abs(registros_lidos - registros_bronze)
        if divergencia > 0:
            problemas.append(f"Divergência contra Bronze: {divergencia} registros")

        # Check 2 — órfãos pós-join
        orfaos = df.filter(F.col("order_status").isNull()).count()
        if orfaos > 0:
            problemas.append(f"Órfãos pós-join: {orfaos} registros")

        # Check 3 — duplicatas na chave primária
        duplicatas = registros_lidos - df.select("order_id", "order_item_id").distinct().count()
        if duplicatas > 0:
            problemas.append(f"Duplicatas na chave primária: {duplicatas}")

        # Check 4 — nulos nas colunas calculadas
        for coluna in ["valor_total_pedido", "rank_item_pedido", "ltv_acumulado"]:
            nulos = df.filter(F.col(coluna).isNull()).count()
            if nulos > 0:
                problemas.append(f"Nulos em {coluna}: {nulos}")

        # Check 5 — valores impossíveis
        if df.filter(F.col("valor_total_pedido") <= 0).count() > 0:
            problemas.append("valor_total_pedido com zero ou negativo")
        if df.filter(F.col("ltv_acumulado") <= 0).count() > 0:
            problemas.append("ltv_acumulado com zero ou negativo")

        # Check 6 — rank mínimo
        rank_min = df.agg(F.min("rank_item_pedido")).collect()[0][0]
        if rank_min != 1:
            problemas.append(f"Rank mínimo inválido: {rank_min}")

        # Decisão final
        tempo = round(time.time() - inicio, 2)

        if problemas:
            motivo = " | ".join(problemas)
            status = "FAILED"
            mensagem = f"Falha nas validações: {motivo}"
            print(f"  FALHA: {motivo}")
        else:
            status = "SUCCESS"
            mensagem = f"Processado com sucesso em {tempo}s"
            print(f"  SUCCESS — {registros_lidos} registros validados em {tempo}s")

        gravar_log(
            execucao_id        = execucao_id,
            tabela             = nome,
            status             = status,
            registros_lidos    = registros_lidos,
            registros_gravados = registros_lidos if status == "SUCCESS" else None,
            tempo_segundos     = tempo,
            mensagem           = mensagem
        )

    except Exception as e:
        tempo = round(time.time() - inicio, 2)
        gravar_log(
            execucao_id    = execucao_id,
            tabela         = nome,
            status         = "FAILED",
            tempo_segundos = tempo,
            mensagem       = f"Erro inesperado: {str(e)}"
        )
        print(f"  ERRO INESPERADO: {e}")

print("Orquestrador Silver carregado!")